# Lab 6: Fake Job Posting Detection

**Task:** detect fraudulent job advertisements.

**Dataset:** Real or Fake Job Postings (Kaggle: https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction), included as `data/fake_job_postings.zip` as downloaded: about 17,900 job adverts, 866 of them fraudulent.

**What we will do:**
1. Load the data and separate structured fields (location, salary, department, employment type, ...) from free text (title, company profile, description, requirements, benefits).
2. Clean missing and empty values, encode categorical fields, and turn the text into TF-IDF features combined with the structured ones.
3. Train logistic regression, Naive Bayes and a random forest.
4. Evaluate with precision, recall, F1 and ROC AUC.
5. Report the words and fields most predictive of fraud, and show the posts the model flags most strongly.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

jobs = pd.read_csv("data/fake_job_postings.zip")      # pandas reads a zip holding a single CSV directly
print(jobs.shape)
print(jobs["fraudulent"].value_counts().to_dict(), f"-> {jobs['fraudulent'].mean():.1%} fraudulent")
jobs.head(2).T

## 1. Structured fields and free text

In [ ]:
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
structured_cols = ["location", "department", "salary_range", "employment_type", "required_experience",
                   "required_education", "industry", "function", "telecommuting", "has_company_logo", "has_questions"]
summary = pd.DataFrame({"missing %": (100 * jobs.isna().mean()).round(1), "distinct values": jobs.nunique()})
summary.loc[structured_cols + text_cols]

Many fields are often empty: the salary range in 84% of adverts, the department in 65%, the company profile in 19%. **Whether a field is missing can itself be informative.** A real company usually describes itself and shows its logo:

In [ ]:
for col in ["company_profile", "salary_range", "requirements", "benefits"]:
    rate = jobs.groupby(jobs[col].isna())["fraudulent"].mean()
    print(f"{col:16s}: fraud rate when present {rate[False]:.3f}, when missing {rate[True]:.3f}")
print(jobs.groupby("has_company_logo")["fraudulent"].mean().round(3).rename("fraud rate by has_company_logo").to_dict())

Adverts without a company profile are fraudulent 18% of the time, against 2% with one; adverts without a logo 16% against 2%. And, perhaps surprisingly, adverts that *do* state a salary are more often fraudulent (7.8% against 4.3%): an attractive salary is a lure. So we keep "missing" as information instead of filling it in.

## 2. Preprocessing

### Structured fields
- **Location** looks like `"US, NY, New York"`: 3,105 distinct values, too many to encode directly. We keep the country and the state.
- **Salary range** looks like `"50000-65000"` (and sometimes contains month names, an Excel import accident). We keep only whether a salary is given, and whether it is a valid numeric range.
- **Department** has 1,337 spellings of free text; we keep only whether it is given.
- The other categorical fields (employment type, experience, education, industry, function) have few values; missing becomes its own category "unknown". One-hot encoding comes later, inside the pipeline.

In [ ]:
df = jobs.copy()
loc = df["location"].fillna("").str.split(",", expand=True)
df["country"] = loc[0].str.strip().replace("", "unknown")
df["state"] = (loc[0].str.strip() + "-" + loc[1].fillna("").str.strip()).replace("-", "unknown")
df["has_salary"] = df["salary_range"].notna().astype(int)
df["salary_is_numeric"] = df["salary_range"].fillna("").str.fullmatch(r"\d+-\d+").astype(int)
df["has_department"] = df["department"].notna().astype(int)
for col in ["company_profile", "requirements", "benefits"]:
    df[f"has_{col}"] = df[col].fillna("").str.strip().ne("").astype(int)
cat_cols = ["country", "state", "employment_type", "required_experience", "required_education", "industry", "function"]
df[cat_cols] = df[cat_cols].fillna("unknown")
flag_cols = ["telecommuting", "has_company_logo", "has_questions", "has_salary", "salary_is_numeric",
             "has_department", "has_company_profile", "has_requirements", "has_benefits"]
df[["country", "state", "has_salary", "salary_is_numeric", "has_company_profile"]].head()

### Free text
We join the five text fields into one document per advert, after cleaning: the descriptions contain HTML leftovers, URLs, e-mail addresses and phone numbers that were replaced by placeholders such as `#URL_...#`. We lowercase, turn those placeholders into single words (`urltoken`, `emailtoken`, `phonetoken`, since the *presence* of contact details can matter), and remove everything that isn't a letter. Empty fields become empty strings.

In [ ]:
def clean_text(t):
    t = t.lower()
    t = re.sub(r"#url_\w+#", " urltoken ", t)
    t = re.sub(r"#email_\w+#", " emailtoken ", t)
    t = re.sub(r"#phone_\w+#", " phonetoken ", t)
    t = re.sub(r"[^a-z]+", " ", t)
    return t.strip()

df["text"] = df[text_cols].fillna("").agg(" ".join, axis=1).map(clean_text)
print("adverts with (almost) no text:", (df["text"].str.len() < 20).sum())
print(df["text"].iloc[0][:300])

**TF-IDF** turns each document into a vector: one entry per word in the vocabulary, the word's count in the document (term frequency) multiplied by its inverse document frequency (large for words that appear in few adverts, small for words in almost every advert), and normalised to length 1. Words in fewer than 5 adverts are ignored (typos, names), as are English stopwords; we use single words and two-word phrases (bigrams).

### Combining everything
A `ColumnTransformer` builds one feature matrix: TF-IDF columns from the text, one-hot columns from the categories, and the 0/1 flags as they are. Everything is fitted on the training data only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder

X = df[["text"] + cat_cols + flag_cols]
y = df["fraudulent"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

def make_features():
    return ColumnTransformer([
        ("text", TfidfVectorizer(stop_words="english", min_df=5, max_df=0.8, ngram_range=(1, 2), sublinear_tf=True), "text"),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ("flags", "passthrough", flag_cols),
    ])
print("features:", make_features().fit(x_train).transform(x_train).shape[1])

## 3. Models

- **Logistic regression** works well with many sparse features; balanced class weights make the 5% fraudulent adverts count as much as the rest.
- **Multinomial Naive Bayes** treats the features as counts (TF-IDF values work too) and is extremely fast.
- **Random forest** on tens of thousands of sparse columns is slower; we let each tree consider a small share of the columns at each split.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

models = {
    "logistic regression": LogisticRegression(C=10, class_weight="balanced", max_iter=3000),
    "naive bayes": MultinomialNB(alpha=0.1),
    "random forest": RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=2,
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=0),
}
results, fitted = [], {}
for name, model in models.items():
    pipe = Pipeline([("features", make_features()), ("model", model)]).fit(x_train, y_train)
    p = pipe.predict_proba(x_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    fitted[name] = (pipe, p)
    results.append({"model": name, "precision": precision_score(y_test, pred), "recall": recall_score(y_test, pred),
                    "F1": f1_score(y_test, pred), "ROC AUC": roc_auc_score(y_test, p), "PR AUC": average_precision_score(y_test, p)})
pd.DataFrame(results).round(3)

## 4. Evaluation

All three rank adverts very well (ROC AUC 0.98–0.99), logistic regression best. At the default cutoff of 0.5 they behave differently: the precision/recall balance depends on the class weighting and on how each model's probabilities are scaled. Naive Bayes probabilities, for example, are usually extreme (close to 0 or 1), because it multiplies evidence from thousands of words as if they were independent.

Choosing the cutoff for each model to maximise F1 shows what each can do at its best balance (in practice choose it on validation data; here we only want to compare):

In [ ]:
from sklearn.metrics import precision_recall_curve

rows = []
for name, (pipe, p) in fitted.items():
    prec, rec, thr = precision_recall_curve(y_test, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    k = np.argmax(f1[:-1])
    rows.append({"model": name, "best-F1 cutoff": thr[k], "precision": prec[k], "recall": rec[k], "F1": f1[k]})
pd.DataFrame(rows).round(3)

In [ ]:
best = max(fitted, key=lambda n: roc_auc_score(y_test, fitted[n][1]))
pipe, p = fitted[best]
print("best model by ROC AUC:", best)
print(pd.DataFrame(confusion_matrix(y_test, (p >= 0.5).astype(int)), index=["real", "fraud"], columns=["predicted real", "predicted fraud"]))

## 5. What predicts fraud?

### Words and fields (logistic regression weights)
Each weight is the change in the log-odds of fraud for a one-unit increase in that feature, holding the others fixed. Positive weights point to fraud.

In [ ]:
lr_pipe = fitted["logistic regression"][0]
names = lr_pipe[0].get_feature_names_out()
weights = pd.Series(lr_pipe[-1].coef_[0], index=names)
pretty = lambda s: s.rename(lambda n: n.split("__", 1)[1])
print("towards FRAUD:\n", pretty(weights.sort_values(ascending=False).head(20)).round(2))
print("\ntowards REAL:\n", pretty(weights.sort_values().head(15)).round(2))

### Fields (random forest importance, structured features only)

In [ ]:
rf_pipe = fitted["random forest"][0]
imp = pd.Series(rf_pipe[-1].feature_importances_, index=rf_pipe[0].get_feature_names_out())
structured_imp = imp[~imp.index.str.startswith("text__")]
print(pretty(structured_imp.sort_values(ascending=False).head(12)).round(4))
print(f"\nshare of total importance: text {imp[imp.index.str.startswith('text__')].sum():.2f}, structured fields {structured_imp.sum():.2f}")

**What the model learned.** The text carries most of the signal (94% of the random forest's importance). Fraudulent adverts tend to:
- ask applicants to **apply through a link** or by phone (`link urltoken`, `apply using`, `phonetoken`) instead of through the job site;
- use **money language**: "earn", "money", "income", and offer clerical work such as **data entry** and assistant jobs;
- lack a **company logo and company profile** (the most important structured fields);
- come from particular industries and places: **oil & energy** jobs in Houston and adverts from **Malaysia** (`country_MY`) stand out, and a few named recruiters ("aptitude staffing") appear in many fraudulent adverts.

Real adverts more often come from internet, health-care and restaurant employers and talk about clients, software and web products, and what is expected of candidates. Several of these cues belong to a few specific scam campaigns in this dataset (many near-identical adverts from one source), so a model trained here would need retraining as scams change.

### Examples the model flags most strongly

In [ ]:
test_view = jobs.loc[x_test.index, ["title", "location", "company_profile", "salary_range", "fraudulent"]].copy()
test_view["fraud probability"] = fitted["logistic regression"][1]
test_view["company_profile"] = test_view["company_profile"].fillna("(none)").str[:60]
print("Top 10 flagged adverts:")
print(test_view.sort_values("fraud probability", ascending=False).head(10).round(3).to_string())

In [ ]:
print("Real adverts with the highest scores (false alarms):")
print(test_view[test_view["fraudulent"] == 0].sort_values("fraud probability", ascending=False).head(5).round(3).to_string())
print("\nFraudulent adverts with the lowest scores (missed):")
print(test_view[test_view["fraudulent"] == 1].sort_values("fraud probability").head(5).round(3).to_string())

The ten most strongly flagged adverts are all frauds: engineering jobs in Houston and home-based "payroll typist / data entry" adverts, none with a company profile. The false alarms are real adverts that look like scams: clerical jobs paid by the hour, no company profile. The missed frauds also have no company profile, but they look like ordinary professional jobs (quant analyst, project manager, big data engineer) without any money language, so the text gives nothing away. A practical system would use the score to send suspicious adverts for human review rather than to delete them automatically.